In [1]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys
import csv
import numpy as np

# تنظیم مسیرها
CODE_DIR = Path.cwd()
HELDOUT_ROOT = Path("/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/heldout_IEEE")  # ویرایش کن
RESULTS_ROOT = CODE_DIR / "results"

CASES = [30, 39, 57]
VARIANTS = ["ddc", "ddc_yb", "ddc_imag", "ddc_real"]
SEED = 42
DEVICE = "cuda"

# پوشهٔ مستقل برای خروجی این ارزیابی
OUT_DIR = CODE_DIR / "heldout_results" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR.mkdir(parents=True)

predict_script = CODE_DIR / "predict_ablation.py"
assert predict_script.is_file(), f"Missing: {predict_script}"

# بررسی همهٔ فایل‌ها پیش از شروع
for case in CASES:
    data_path = HELDOUT_ROOT / f"case{case}" / "dataset_heldout.npz"
    assert data_path.is_file(), f"Missing: {data_path}"

    for variant in VARIANTS:
        checkpoint = (
            RESULTS_ROOT / f"case{case}_dq"
            / "axial" / variant / f"seed_{SEED}" / "best.pt"
        )
        assert checkpoint.is_file(), f"Missing: {checkpoint}"

rows = []

for case in CASES:
    data_path = HELDOUT_ROOT / f"case{case}" / "dataset_heldout.npz"

    with np.load(data_path, allow_pickle=False) as data:
        target = data["DQ_raw"].astype(np.float64)
        source_indices = data["source_indices"]

    for variant in VARIANTS:
        checkpoint = (
            RESULTS_ROOT / f"case{case}_dq"
            / "axial" / variant / f"seed_{SEED}" / "best.pt"
        )
        prediction_path = OUT_DIR / f"case{case}_{variant}_seed{SEED}.npz"

        # بارگذاری checkpoint، کنترل نرمال‌سازی و پیش‌بینی در مقیاس اصلی DQ
        subprocess.run(
            [
                sys.executable, "-u", str(predict_script.resolve()),
                "--checkpoint", str(checkpoint.resolve()),
                "--input", str(data_path.resolve()),
                "--output", str(prediction_path.resolve()),
                "--device", DEVICE,
                "--batch-size", "16",
            ],
            cwd=CODE_DIR,
            check=True,
        )

        with np.load(prediction_path, allow_pickle=False) as result:
            prediction = result["prediction"].astype(np.float64)

        assert prediction.shape == target.shape
        assert np.isfinite(prediction).all()

        error = prediction - target
        sse = np.sum(error**2)
        sst = np.sum((target - target.mean())**2)

        row = {
            "case": case,
            "model": "axial",
            "variant": variant,
            "seed": SEED,
            "samples": len(target),
            "r2": float(1 - sse / sst) if sst > 0 else None,
            "rmse": float(np.sqrt(np.mean(error**2))),
            "mae": float(np.mean(np.abs(error))),
        }
        rows.append(row)
        print("HELDOUT:", row, flush=True)

        np.savez_compressed(
            prediction_path,
            prediction=prediction,
            target=target,
            source_indices=source_indices,
        )

summary_path = OUT_DIR / "heldout_metrics.csv"
with summary_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("\nSaved:", summary_path)

Saved physical-unit predictions (10, 1, 30, 30) /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/heldout_results/20261107_221630_134651/case30_ddc_seed42.npz
HELDOUT: {'case': 30, 'model': 'axial', 'variant': 'ddc', 'seed': 42, 'samples': 10, 'r2': 0.8824514936667821, 'rmse': 1.266512928669118, 'mae': 0.5051479977274408}
Saved physical-unit predictions (10, 1, 30, 30) /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/heldout_results/20261107_221630_134651/case30_ddc_yb_seed42.npz
HELDOUT: {'case': 30, 'model': 'axial', 'variant': 'ddc_yb', 'seed': 42, 'samples': 10, 'r2': 0.8910883069986449, 'rmse': 1.2190971739352534, 'mae': 0.48284077944740017}
Saved physical-unit predictions (10, 1, 30, 30) /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/heldout_results/20261107_221630_134651/case30_ddc_imag_seed42.npz
HELDOUT: {'case': 30, 'model': 'axial', 'variant': 'ddc_imag', 'seed': 4